# Montana 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for Montana, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals).

**Output**: A single CSV where each row is a county and columns include:

- Primary per-candidate vote counts (prefixed with `pri_`)
- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_primary_total`, `dem_primary_total`, `rep_general_total`, `dem_general_total`, `lib_general_total`, `cst_general_total`, `ind_general_total`

**Last Updated**: 2025/10/22

## 0. Library Import

In [12]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Inputs & Parameters

Define raw file paths once here so the entire notebook is easy to rerun on another machine. If a path changes, we only update it here. We keep a single `OUTPUT_PATH` so all exports land in one known place.

In [27]:
# MT 2008 dataset path
PRIMARY_PATH = r"../../data/raw/2008/MT/20080603__mt__primary__county.csv"
GENERAL_PATH = r"../../data/raw/2008/MT/20081104__mt__general__county.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/MT/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### a. Primary Election Dataset

In [14]:
# Load primary data
primary_df = pd.read_csv(PRIMARY_PATH)
primary_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
0,Beaverhead,President,NaN,Democrat,Hillary Clinton,413
1,Big Horn,President,NaN,Democrat,Hillary Clinton,567
2,Blaine,President,NaN,Democrat,Hillary Clinton,568
3,Broadwater,President,NaN,Democrat,Hillary Clinton,386
4,Carbon,President,NaN,Democrat,Hillary Clinton,942
5,Carter,President,NaN,Democrat,Hillary Clinton,52
6,Cascade,President,NaN,Democrat,Hillary Clinton,7290
7,Chouteau,President,NaN,Democrat,Hillary Clinton,397
8,Custer,President,NaN,Democrat,Hillary Clinton,1084
9,Daniels,President,NaN,Democrat,Hillary Clinton,208


In [15]:
# Different values in 'office' column
primary_df["office"].value_counts()

office
State House                             602
U.S. Senate                             392
President                               336
Governor                                280
Attorney General                        280
Superintendent of Public Instruction    280
U.S. House                              224
State Senate                            180
State Auditor                           112
Secretary of State                      112
Name: count, dtype: int64

In [16]:
# Only keep rows where 'office' is 'President'
primary_df = primary_df[primary_df["office"] == "President"]
primary_df.shape

(336, 6)

In [17]:
# Number of missing values in each column
primary_df.isna().sum()

county         0
office         0
district     336
party          0
candidate      0
votes          0
dtype: int64

Here, we can see that `office` has only one value "president" while `district` does not have any values at all. Thus, we can drop these two columns given that they don't give any additional information for our analysis.

In [18]:
# Drop "office" and "district" columns
primary_df = primary_df.drop(columns=["office", "district"]).reset_index(drop=True)
primary_df.shape

(336, 4)

In [19]:
# List out all the parties in the primary election data
primary_df["party"].value_counts()

party
Democrat      168
Republican    168
Name: count, dtype: int64

In [20]:
# Different candidate in primary election data
primary_df["candidate"].value_counts()

candidate
No Preference      112
Hillary Clinton     56
Barack Obama        56
John McCain         56
Ron Paul            56
Name: count, dtype: int64

There is an interesting value of "No Preference" in `candidate`. We might want to have a closer look on observations with this value in such colum first before deciding what to do with it.

In [21]:
# Sneak peek on rows with "candidate" == "No Preference"
primary_df[primary_df["candidate"] == "No Preference"].head(DISPLAY_ROWS)

,county,party,candidate,votes
112,Beaverhead,Democrat,No Preference,26
113,Big Horn,Democrat,No Preference,38
114,Blaine,Democrat,No Preference,56
115,Broadwater,Democrat,No Preference,45
116,Carbon,Democrat,No Preference,47
117,Carter,Democrat,No Preference,6
118,Cascade,Democrat,No Preference,604
119,Chouteau,Democrat,No Preference,17
120,Custer,Democrat,No Preference,55
121,Daniels,Democrat,No Preference,27


In [22]:
# Number of missing values now in primary_df
primary_df.isna().sum()

county       0
party        0
candidate    0
votes        0
dtype: int64

Apparently, there is still an associated `party` that comes with each observation where `candidate == "No Preference"`. We wil keep this until we count the total votes for each party later, then drop them eventually.

In [35]:
# Data type of each column in primary_df
primary_df.dtypes

county       object
party        object
candidate    object
votes         int64
dtype: object

In [14]:
# Final look at the cleaned primary_df
primary_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Beaverhead,Democrat,Hillary Clinton,413
1,Big Horn,Democrat,Hillary Clinton,567
2,Blaine,Democrat,Hillary Clinton,568
3,Broadwater,Democrat,Hillary Clinton,386
4,Carbon,Democrat,Hillary Clinton,942
5,Carter,Democrat,Hillary Clinton,52
6,Cascade,Democrat,Hillary Clinton,7290
7,Chouteau,Democrat,Hillary Clinton,397
8,Custer,Democrat,Hillary Clinton,1084
9,Daniels,Democrat,Hillary Clinton,208


In [23]:
# Shape after preprocessing
primary_df.shape

(336, 4)

### b. General Election Dataset

In [24]:
# Load general data
general_df = pd.read_csv(GENERAL_PATH)
general_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
0,Beaverhead,President,NaN,Constitution,Ron Paul,86
1,Big Horn,President,NaN,Constitution,Ron Paul,37
2,Blaine,President,NaN,Constitution,Ron Paul,53
3,Broadwater,President,NaN,Constitution,Ron Paul,12
4,Carbon,President,NaN,Constitution,Ron Paul,130
5,Carter,President,NaN,Constitution,Ron Paul,25
6,Cascade,President,NaN,Constitution,Ron Paul,502
7,Chouteau,President,NaN,Constitution,Ron Paul,53
8,Custer,President,NaN,Constitution,Ron Paul,65
9,Daniels,President,NaN,Constitution,Ron Paul,20


In [25]:
# Different values in 'office' column
general_df["office"].value_counts()

office
State House                             492
President                               280
U.S. House                              168
Governor                                168
Secretary of State                      168
Superintendent of Public Instruction    168
State Senate                            136
U.S. Senate                             112
Attorney General                        112
State Auditor                           112
Name: count, dtype: int64

In [28]:
# Only keep rows where 'office' is 'President'
general_df = general_df[general_df["office"] == "President"]
general_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
0,Beaverhead,President,NaN,Constitution,Ron Paul,86
1,Big Horn,President,NaN,Constitution,Ron Paul,37
2,Blaine,President,NaN,Constitution,Ron Paul,53
3,Broadwater,President,NaN,Constitution,Ron Paul,12
4,Carbon,President,NaN,Constitution,Ron Paul,130
5,Carter,President,NaN,Constitution,Ron Paul,25
6,Cascade,President,NaN,Constitution,Ron Paul,502
7,Chouteau,President,NaN,Constitution,Ron Paul,53
8,Custer,President,NaN,Constitution,Ron Paul,65
9,Daniels,President,NaN,Constitution,Ron Paul,20


In [29]:
# Primary data shape when only considering President/VicePresident
general_df.shape

(280, 6)

In [30]:
# Number of missing values in each column
general_df.isna().sum()

county         0
office         0
district     280
party          0
candidate      0
votes          0
dtype: int64

In [31]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the "district" column since it's all missing values
general_df = general_df.drop(columns=["office", "district"]).reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Beaverhead,Constitution,Ron Paul,86
1,Big Horn,Constitution,Ron Paul,37
2,Blaine,Constitution,Ron Paul,53
3,Broadwater,Constitution,Ron Paul,12
4,Carbon,Constitution,Ron Paul,130
5,Carter,Constitution,Ron Paul,25
6,Cascade,Constitution,Ron Paul,502
7,Chouteau,Constitution,Ron Paul,53
8,Custer,Constitution,Ron Paul,65
9,Daniels,Constitution,Ron Paul,20


In [32]:
# Candidates in general_df
general_df["candidate"].value_counts()

candidate
Ron Paul        56
Barack Obama    56
Ralph Nader     56
Bob Barr        56
John McCain     56
Name: count, dtype: int64

In [33]:
# List out all the parties in the general election data
general_df["party"].value_counts()

party
Constitution    56
Democrat        56
Independent     56
Libertarian     56
Republican      56
Name: count, dtype: int64

In [34]:
# Data type of each column in general_df
general_df.dtypes

county       object
party        object
candidate    object
votes         int64
dtype: object

In [36]:
# Final look at the cleaned general_df
general_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Beaverhead,Constitution,Ron Paul,86
1,Big Horn,Constitution,Ron Paul,37
2,Blaine,Constitution,Ron Paul,53
3,Broadwater,Constitution,Ron Paul,12
4,Carbon,Constitution,Ron Paul,130
5,Carter,Constitution,Ron Paul,25
6,Cascade,Constitution,Ron Paul,502
7,Chouteau,Constitution,Ron Paul,53
8,Custer,Constitution,Ron Paul,65
9,Daniels,Constitution,Ron Paul,20


In [37]:
# Shape after preprocessing
general_df.shape

(280, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [38]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: Democratic -> dem, Republican -> rep
    """
    return(s.str.strip()
           .str.capitalize()
           .map({
                "Democrat"       : "dem", 
                "Republican"     : "rep",
                "Constitution"   : "cst",
                "Libertarian"    : "lib",
                "Independent"    : "ind"
               })
           .fillna(s.str.strip().str.lower()))      # For defensive purposes only, would not expect other parties

In [39]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [40]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [41]:
# Primary dataframe pivot
primary_pivot = pivot_wide(primary_df, prefix="pri")
primary_pivot.head(DISPLAY_ROWS)

,county,pri_dem_CLINTON,pri_dem_OBAMA,pri_dem_PREFERENCE,pri_rep_MCCAIN,pri_rep_PAUL,pri_rep_PREFERENCE
0,Beaverhead,413,771,26,1385,285,55
1,Big Horn,567,2156,38,441,70,9
2,Blaine,568,700,56,308,89,11
3,Broadwater,386,456,45,748,124,18
4,Carbon,942,1252,47,905,259,28
5,Carter,52,70,6,215,86,13
6,Cascade,7290,7892,604,4893,857,95
7,Chouteau,397,441,17,860,212,78
8,Custer,1084,862,55,1013,133,19
9,Daniels,208,168,27,209,71,7


Note that there are two column with no preference as value in `candidate` that still had votes (`pri_dem_PREFERENCE` and `pri_rep_PREFERENCE`). We will keep those for total counting purposes and drop them at the end.

In [42]:
# Primary dataframe shape after pivot
primary_pivot.shape

(56, 7)

In [43]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_cst_PAUL,gen_dem_OBAMA,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN
0,Beaverhead,86,1617,38,14,3008
1,Big Horn,37,3516,33,4,1628
2,Blaine,53,1702,22,8,1139
3,Broadwater,12,365,10,6,756
4,Carbon,130,2443,45,20,3108
5,Carter,25,111,2,4,573
6,Cascade,502,17664,286,79,16857
7,Chouteau,53,1122,44,8,1634
8,Custer,65,2267,54,19,3047
9,Daniels,20,343,14,2,694


In [44]:
# General dataframe shape after pivot
general_pivot.shape

(56, 6)

## 4. Merge Dataframes

Before merging, we verify that county names match across primary and general:

In [45]:
# Check if county names match between primary_df and general_df
primary_counties = set(primary_pivot["county"].unique())
general_counties = set(general_pivot["county"].unique())
common_counties = primary_counties.intersection(general_counties)
print(f"Number of common counties: {len(common_counties)} out of {len(primary_counties)}")

Number of common counties: 56 out of 56


Great. Since we know that all counties name are matched, we don't need to perform further data preprocessing to match the county names. Thus, we can now merge them:

In [46]:
# Merge primary and general dataframes on 'county'
merged_df = primary_pivot.merge(general_pivot, on="county", how="inner").fillna(0)    # There should be no missing values to fill with 0
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_CLINTON,pri_dem_OBAMA,pri_dem_PREFERENCE,pri_rep_MCCAIN,pri_rep_PAUL,pri_rep_PREFERENCE,gen_cst_PAUL,gen_dem_OBAMA,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN
0,Beaverhead,413,771,26,1385,285,55,86,1617,38,14,3008
1,Big Horn,567,2156,38,441,70,9,37,3516,33,4,1628
2,Blaine,568,700,56,308,89,11,53,1702,22,8,1139
3,Broadwater,386,456,45,748,124,18,12,365,10,6,756
4,Carbon,942,1252,47,905,259,28,130,2443,45,20,3108
5,Carter,52,70,6,215,86,13,25,111,2,4,573
6,Cascade,7290,7892,604,4893,857,95,502,17664,286,79,16857
7,Chouteau,397,441,17,860,212,78,53,1122,44,8,1634
8,Custer,1084,862,55,1013,133,19,65,2267,54,19,3047
9,Daniels,208,168,27,209,71,7,20,343,14,2,694


In [47]:
# Statistics check on merged dataframe 
merged_df.describe()

,pri_dem_CLINTON,pri_dem_OBAMA,pri_dem_PREFERENCE,pri_rep_MCCAIN,pri_rep_PAUL,pri_rep_PREFERENCE,gen_cst_PAUL,gen_dem_OBAMA,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN
count,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000
mean,1337.303571,1842.392857,77.821429,1299.839286,367.964286,41.660714,189.964286,4136.910714,65.821429,24.196429,4335.053571
std,2269.416705,3345.494907,123.882658,1924.530089,615.638470,57.328805,336.019207,7680.577718,103.968545,37.356961,7167.925077
min,38.000000,27.000000,0.000000,66.000000,22.000000,0.000000,3.000000,68.000000,1.000000,1.000000,227.000000
25%,205.500000,210.250000,13.750000,305.000000,76.250000,10.750000,32.500000,366.500000,12.500000,4.750000,801.750000
50%,483.500000,641.500000,37.500000,575.000000,137.500000,18.000000,62.000000,1403.000000,32.500000,11.000000,1608.000000
75%,1017.250000,1284.500000,70.000000,1201.250000,286.250000,49.750000,130.750000,3119.250000,55.250000,20.250000,3272.500000
max,11544.000000,16749.000000,604.000000,9997.000000,3691.000000,275.000000,1683.000000,36531.000000,514.000000,162.000000,36483.000000


Now, we will add party totals columns: 

- Primary totals:
    * `rep_primary_total` = sum of all `pri_rep_*` columns
    * `dem_primary_total` = sum of all `pri_dem_*` columns

- General totals:
    * `rep_general_total` = sum of all `gen_rep_*` columns
    * `dem_general_total` = sum of all `gen_dem_*` columns
    * `lib_general_total` = sum of all `gen_lib_*` columns
    * `ind_general_total` = sum of all `gen_ind_*` columns
    * `cst_general_total` = sum of all `gen_cst_*` columns

In [49]:
# Add party totals for primary election
rep_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_rep_")]
dem_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_dem_")]

merged_df["rep_primary_total"] = merged_df[rep_primary_cols].sum(axis=1) if rep_primary_cols else 0
merged_df["dem_primary_total"] = merged_df[dem_primary_cols].sum(axis=1) if dem_primary_cols else 0

Now, we have calculated the total vote for each party. Thus, we can drop the two PREFERENCE (as in "No Preference") columns.

In [50]:
# Drop PREFERENCE ("No Preference") columns for primary election
merged_df = merged_df.drop(columns=["pri_dem_PREFERENCE", "pri_rep_PREFERENCE"])

# Snippet at the merged dataframe with primary totals
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_CLINTON,pri_dem_OBAMA,pri_rep_MCCAIN,pri_rep_PAUL,gen_cst_PAUL,gen_dem_OBAMA,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN,rep_primary_total,dem_primary_total
0,Beaverhead,413,771,1385,285,86,1617,38,14,3008,1725,1210
1,Big Horn,567,2156,441,70,37,3516,33,4,1628,520,2761
2,Blaine,568,700,308,89,53,1702,22,8,1139,408,1324
3,Broadwater,386,456,748,124,12,365,10,6,756,890,887
4,Carbon,942,1252,905,259,130,2443,45,20,3108,1192,2241
5,Carter,52,70,215,86,25,111,2,4,573,314,128
6,Cascade,7290,7892,4893,857,502,17664,286,79,16857,5845,15786
7,Chouteau,397,441,860,212,53,1122,44,8,1634,1150,855
8,Custer,1084,862,1013,133,65,2267,54,19,3047,1165,2001
9,Daniels,208,168,209,71,20,343,14,2,694,287,403


In [51]:
# Add party totals for general election
rep_general_cols   = [c for c in merged_df.columns if c.startswith("gen_rep_")]
dem_general_cols   = [c for c in merged_df.columns if c.startswith("gen_dem_")]
lib_general_cols   = [c for c in merged_df.columns if c.startswith("gen_lib_")]
cst_general_cols   = [c for c in merged_df.columns if c.startswith("gen_cst_")]
ind_general_cols   = [c for c in merged_df.columns if c.startswith("gen_ind_")]

merged_df["rep_general_total"] = merged_df[rep_general_cols].sum(axis=1) if rep_general_cols else 0
merged_df["dem_general_total"] = merged_df[dem_general_cols].sum(axis=1) if dem_general_cols else 0
merged_df["lib_general_total"] = merged_df[lib_general_cols].sum(axis=1) if lib_general_cols else 0
merged_df["cst_general_total"] = merged_df[cst_general_cols].sum(axis=1) if cst_general_cols else 0
merged_df["ind_general_total"] = merged_df[ind_general_cols].sum(axis=1) if ind_general_cols else 0

In [52]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned dataframe:")
merged_df.columns

Final columns in the cleaned dataframe:


Index(['county', 'pri_dem_CLINTON', 'pri_dem_OBAMA', 'pri_rep_MCCAIN',
       'pri_rep_PAUL', 'gen_cst_PAUL', 'gen_dem_OBAMA', 'gen_ind_NADER',
       'gen_lib_BARR', 'gen_rep_MCCAIN', 'rep_primary_total',
       'dem_primary_total', 'rep_general_total', 'dem_general_total',
       'lib_general_total', 'cst_general_total', 'ind_general_total'],
      dtype='object')

In [53]:
# Preview merged dataframe with totals
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_CLINTON,pri_dem_OBAMA,pri_rep_MCCAIN,pri_rep_PAUL,gen_cst_PAUL,gen_dem_OBAMA,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN,rep_primary_total,dem_primary_total,rep_general_total,dem_general_total,lib_general_total,cst_general_total,ind_general_total
0,Beaverhead,413,771,1385,285,86,1617,38,14,3008,1725,1210,3008,1617,14,86,38
1,Big Horn,567,2156,441,70,37,3516,33,4,1628,520,2761,1628,3516,4,37,33
2,Blaine,568,700,308,89,53,1702,22,8,1139,408,1324,1139,1702,8,53,22
3,Broadwater,386,456,748,124,12,365,10,6,756,890,887,756,365,6,12,10
4,Carbon,942,1252,905,259,130,2443,45,20,3108,1192,2241,3108,2443,20,130,45
5,Carter,52,70,215,86,25,111,2,4,573,314,128,573,111,4,25,2
6,Cascade,7290,7892,4893,857,502,17664,286,79,16857,5845,15786,16857,17664,79,502,286
7,Chouteau,397,441,860,212,53,1122,44,8,1634,1150,855,1634,1122,8,53,44
8,Custer,1084,862,1013,133,65,2267,54,19,3047,1165,2001,3047,2267,19,65,54
9,Daniels,208,168,209,71,20,343,14,2,694,287,403,694,343,2,20,14


Now, we save the cleaned dataframe into the processed directory.

In [54]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(OUTPUT_PATH + "MT.csv", index=False)